In [24]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import classification_report
import pandas as pd
import requests
import re
import numpy as np
import warnings
import os
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')


def get_answers(text):
    if re.fullmatch("\d\d?", text):
        matches = re.findall("\d\d?", text)
    elif re.fullmatch("[\d\d?, ]+", text):
        matches = re.findall("\d\d?", text)
    else:
        matches = re.findall("\d\d?", text)[-1:]
    return [int(m) for m in matches if int(m) < 15]



def default_evaluation(true_labels, pred_labels):
    result = dict()
    result['precision'] = precision_score(true_labels, pred_labels)
    result['recall'] = recall_score(true_labels, pred_labels)
    result['f1'] = f1_score(true_labels, pred_labels)
    result['accuracy'] = accuracy_score(true_labels, pred_labels)
    return result


def get_sampled_results(preds, df, report=False, include_pop=False):
    new_df = {"sample_id": [], "questionEntityId":[],"prediction": []}
    if include_pop:
        new_df["popularity"] = []

    for _, row in preds.iterrows():
        for _, id in row['match_dict'].items():
            new_df["sample_id"].append(id)
            new_df["prediction"].append(id in row['prediction_sample_id'])
            new_df["questionEntityId"].append(row['questionEntityId'])
            if include_pop:
                # print(type(row["popularity"]))
                new_df["popularity"].append(row["popularity"][0] if type(row["popularity"]) == list else row["popularity"])

    result = pd.DataFrame(data=new_df)
    result = result.sort_values(by=["sample_id"]).reset_index(drop=True)
    result.prediction = result.prediction.astype(np.int32)
    all_sample_ids = set(result["sample_id"].values.tolist())
    df = df[df.sample_id.apply(lambda x: x in all_sample_ids)].sort_values(by=["sample_id"])
    result["correct"] = df['correct'].astype(np.int32).values
    # if include_pop:
        # result["popularity"] = result.sample_id.apply(lambda x: sample2pop(df, x))
    
    if report:
        true_labels = df['correct'].astype(np.int32).values
        pred_labels = result['prediction'].astype(np.int32).values
        print(classification_report(pred_labels, true_labels))
        metrics = default_evaluation(true_labels, pred_labels)
        print(metrics)
        return result, metrics
    
    return result


key = ''
def wikimedia_pagereviews(label, start_data="20230101", end_date="20230901"):
    label = label[0].upper() + label[1:]
    if '/' in label:
        test_lbl = label.replace('/', '%2F')
    elif '?' in label:
        test_lbl = label.replace('?', '%3F')
    else:
        test_lbl = '_'.join(label.split(' '))
        
    url = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia.org/all-access/all-agents/"
    url += f"{test_lbl}/monthly/{start_data}/{end_date}"
    headers = {
        'accept': 'application/json',
        'Authorization': key,
        'User-Agent': '2368ba8df0e5f6b69af8027312c507c654c234c8'
    }

    response = requests.get(
        url = url,
        headers=headers,
    )   
    print(url)
    
    return response.json()['items'][0]['views']

def sample2wiki_id(df, sample_id):
    sample_id = int(sample_id)
    return df[df['sample_id'] == sample_id]['answerEntityId'].values[0]

def sample2wiki(df, sample_id):
    sample_id = int(sample_id)
    return df[df['sample_id'] == sample_id]['answerEntity'].values[0]

def sample2pop(df, sample_id):
    sample_id = int(sample_id)
    answer_id = sample2wiki_id(df, sample_id)
    pop = views[views['answerEntityId'] == answer_id]['views'].values
    return pop[0] if len(pop) > 0 else -1

def index2sample(match_dict, idx):
    return match_dict[idx]




def popularity(df, raw):
    res = []
    for token in raw['top_token']:
        try:
            sample_id = index2sample(raw['match_dict'], token)
        except:
            continue
        try:
            wiki_id = sample2wiki_id(df, sample_id)
            val = views[views['answerEntityId'] == wiki_id]['views'].values
        except:
            wiki = sample2wiki(df, sample_id)
            val = views[views['answerEntity'] == wiki]['views'].values
        if val:
            res.append(val[0])
    return sum(res) / len(res) if res else -1


def init_preds(preds, multi=False, first=False):
    preds['prediction'][preds['prediction'].isna()] = "None"
    preds['raw_prediction'] = preds["prediction"]
    preds['match_dict'] = preds['match_dict'].apply(eval)
    preds['top_token'] = preds['raw_prediction'].apply(get_answers)
    preds['popularity'] = [popularity(df, row) for i, row in preds.iterrows()]
    preds['popularity'] = [popularity(df, row) for i, row in preds.iterrows()]
    preds['prediction_sample_id'] = preds.apply(lambda row: [index2sample(row["match_dict"], int(token)) for token in row["top_token"] if int(token) in row["match_dict"]], axis=1)

In [13]:
train_file_path = "../TextGraphs17-shared-task/data/tsv/train.tsv"
file_ws = '../analysis/llama_8B_train_sample_ds_ws_probs.tsv'
file_ds = '../analysis/llama_8B_train_sample_ds_probs.tsv'

df = pd.read_csv(train_file_path, sep="\t")
views = pd.read_csv("views.csv")

df_ws = pd.read_csv(file_ws, sep='\t')
init_preds(df_ws)

df_ds = pd.read_csv(file_ds, sep='\t')
init_preds(df_ds)

In [25]:
anal_dir = "../analysis/"
all_files = os.listdir(anal_dir)
data = {}

for file in all_files:
    print(f"Loading {file}")
    name = file.split(".")[0]
    filepath = os.path.join(anal_dir, file)
    df_ = pd.read_csv(filepath, sep='\t')
    init_preds(df_)
    data[name] = df_

Loading llama_70B_train_sample_ds_probs.tsv
Loading llama_70B_train_sample_ds_tg_probs.tsv
Loading llama_70B_train_sample_ds_tg_ws_probs.tsv
Loading llama_70B_train_sample_ds_ws_probs.tsv
Loading llama_8B_train_sample_ds_probs.tsv
Loading llama_8B_train_sample_ds_tg_probs.tsv
Loading llama_8B_train_sample_ds_tg_ws_probs.tsv
Loading llama_8B_train_sample_ds_ws_probs.tsv


In [26]:
top = 100
for name, df_ in data.items():
    df_preds = get_sampled_results(df_, df, report=False, include_pop=True)
    df_preds = df_preds[df_preds.popularity != -1].sort_values("popularity", ascending=False).reset_index(drop=True)
    print(f"-----------{name} top {top}----------------")
    df_slice = df_preds.iloc[0:top]
    metrics = default_evaluation(df_slice["prediction"], df_slice["correct"])
    for k, v in metrics.items():
        print(f"{k}: {v}")
    print(f"-----------{name} tail {top}----------------")
    df_slice = df_preds.iloc[-top:]
    metrics = default_evaluation(df_slice["prediction"], df_slice["correct"])
    for k, v in metrics.items():
        print(f"{k}: {v}")

-----------llama_70B_train_sample_ds_probs top 100----------------
precision: 0.75
recall: 0.75
f1: 0.75
accuracy: 0.96
-----------llama_70B_train_sample_ds_probs tail 100----------------
precision: 0.7777777777777778
recall: 0.7777777777777778
f1: 0.7777777777777778
accuracy: 0.96
-----------llama_70B_train_sample_ds_tg_probs top 100----------------
precision: 0.7
recall: 0.7777777777777778
f1: 0.7368421052631579
accuracy: 0.95
-----------llama_70B_train_sample_ds_tg_probs tail 100----------------
precision: 0.6
recall: 0.6666666666666666
f1: 0.631578947368421
accuracy: 0.93
-----------llama_70B_train_sample_ds_tg_ws_probs top 100----------------
precision: 0.6
recall: 0.6
f1: 0.6
accuracy: 0.92
-----------llama_70B_train_sample_ds_tg_ws_probs tail 100----------------
precision: 0.6363636363636364
recall: 0.7777777777777778
f1: 0.7
accuracy: 0.94
-----------llama_70B_train_sample_ds_ws_probs top 100----------------
precision: 0.8
recall: 0.8
f1: 0.8
accuracy: 0.96
-----------llama_70B